<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-10-tuning-and-evaluation/lesson-10.6-reasoning-moe/notebooks/GCP_Capstone_10.6_ReasoningMoE.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.6 Teaching a Model to Reason — A Reward Written on the Contract
**Netsetos GenAI Engineering — GCP Capstone** · Module 10 · rebuilt on the live lane, 9 September 2026

Module 10's climax. Two ways to make a model reason better, one question that decides which, and a reward function you will watch get hacked in front of you - in the lane's own grammar. The rewards score ModelDraft's JSON; the judge of a citation is `resolve()`, the same function the API uses, over chunks `retrieve()` actually returned; the grounding reward is the golden set through the gate's `normalise()`. Then MoE and gpt-oss-20b, the stop-token bug and the kit's fix for it, and the honest comparison with the instruments the lane has.

*One naming note:* `gpt-oss-20b` is OpenAI's open-weight model. It appears here because it is the mixture-of-experts model that fine-tunes on hardware you can actually rent.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "rag-production-hardening"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson

print("kit:", KIT, "| API:", API_URL, "| datasets:", f"gs://{DATASETS}/sft/")


## Cell 1: The API


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, CALLED THE WAY THE UI CALLS IT: one ID token per request, minted AS the roster member,
# audience = the API (7.3's hour-long fuse never arms). The kit mints it (documind_tools._id_token).
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). They
# land in Cloud Logging first (the sink copies them to BigQuery); this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

def gcs_text(uri: str) -> str:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_text()

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: make_trainset, judge, tune, run_eval
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules: cache_manager, router, breakers, cost
print("helpers: api(), usage_rows(), gcs_text(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: Two ways, one deciding question


In [ ]:
# There are exactly two ways to make a model reason better, and they are
# easier to tell apart than the names suggest.
#
#   DISTILLATION   Show it a better model's working, and have it copy that.
#                  You need examples of good reasoning. This is still SFT -
#                  the same machinery as 10.5, different data.
#
#   GRPO           Let it try, score each attempt, and push it towards the
#                  ones that scored well. You need a SCORER, not examples.
#
# The choice is decided by one question, and it is not about model quality:
def which_approach(can_you_check_the_answer: bool,
                   do_you_have_good_examples: bool) -> str:
    if can_you_check_the_answer and do_you_have_good_examples:
        return 'either - distil first (fast), then GRPO on top (better)'
    if can_you_check_the_answer:
        return 'GRPO - if you can score it, let the model find its own way there'
    if do_you_have_good_examples:
        return 'Distillation - copy the working from a model that already does it'
    return 'neither yet - get one or the other first, or you are guessing'


for check, examples in ((True, True), (True, False), (False, True), (False, False)):
    print(f'  can check={str(check):5} have examples={str(examples):5} -> '
          f'{which_approach(check, examples)}')

print()
print('  DocuMind can CHECK its answers: a citation either points at a chunk that')
print('  exists and supports the claim, or it does not. That is a scorer, so')
print('  DocuMind uses GRPO - and step 5 shows what goes wrong when the scorer')
print('  is not as clever as the model it is scoring.')


## Cell 3: Distillation - copy the working
A reasoning example is question → working → answer. The working is what transfers.


In [ ]:
# Distillation, plainly: copy the working, not just the answer.
#
# A normal SFT example is question -> answer. A reasoning example is
# question -> WORKING -> answer, and the working is the part that transfers.
#
# Two details make it work, and both are easy to get wrong:
#
#   completion-only loss   Train on the assistant's words only, not the
#                          question. Otherwise the model spends its capacity
#                          learning to predict questions, which nobody asked
#                          for.
#   budget forcing         Make the thinking a fixed length. Left alone, a
#                          distilled model either stops thinking too early or
#                          rambles - and rambling is billed per token.
EXAMPLE = {
    'question': 'A senior engineer resigns on 1 March. What is their last working day?',
    'working': ('The HR policy sets the notice period for a senior engineer at 60 days '
                '[Source 1]. Counting 60 days from 1 March gives 30 April.'),
    'answer': '30 April. [Source 1]',
}
print('question:', EXAMPLE['question'])
print('working :', EXAMPLE['working'])
print('answer  :', EXAMPLE['answer'])
print()

# Public reasoning sets exist to start from - s1K-1.1 is about a thousand
# carefully chosen hard problems; OpenThoughts3 is much larger. Take a SLICE.
# You are teaching a format, not a curriculum.
def mask_prompt(example: dict) -> dict:
    """Completion-only loss: the prompt is context, only the rest is trained on."""
    prompt = f"Question: {example['question']}"
    completion = f"{example['working']}\n\nAnswer: {example['answer']}"
    return {'prompt': prompt, 'completion': completion}


m = mask_prompt(EXAMPLE)
print('trained on   :', repr(m['completion'][:60] + '...'))
print('NOT trained on:', repr(m['prompt']))
print()
print('If you train on the whole string, the model gets better at writing')
print('questions about notice periods. That is a real outcome and a useless one.')


## Cell 4: GRPO - write a scorer, not examples
The rewards, on the contract: `resolve()` over the chunks `retrieve()` returned.


In [ ]:
from shared.documind_schemas import ModelDraft, resolve
from run_eval import normalise

# GRPO, PLAINLY: let it try, score the tries, prefer the good ones. You write a function that gives each
# attempt a number; the trainer generates several attempts per question and nudges the model towards the
# ones that scored higher. The signature is trl's, and **kwargs is required even if unused:
#     reward_func(prompts, completions, **kwargs) -> list[float]
# On the lane a completion is ModelDraft's JSON, and the judge of a citation is the SAME function the API
# uses: resolve() keeps a [Source N] only when N is a chunk the model was actually shown, against the chunks
# retrieve() returned for this question. A reward the API would not render is not a reward.
QUESTION = "What is the notice period for a confirmed E3?"
got = documind_tools.retrieve(QUESTION, tenant_id=TENANT, brain="direct")
PACKED = got.get("citations") or []
assert PACKED, "retrieve() returned no chunks: is the lane up and the corpus ingested (make ingest-corpus)?"
print(f"retrieved {len(PACKED)} chunks for the question; [Source 1] = {PACKED[0].get('chunk_id')}")

def draft(c: str) -> ModelDraft | None:
    try:
        return ModelDraft.model_validate_json(c)
    except Exception:
        return None

def format_reward(prompts, completions, **kwargs) -> list[float]:
    """Cheap, mechanical: does the completion LOOK like a DocuMind answer? Parses as ModelDraft, and short."""
    out = []
    for c in completions:
        d = draft(c)
        out.append((0.5 if d else 0.0) + (0.5 if d and len(d.answer.split()) <= 60 else 0.0))
    return out

def citation_validity_reward(prompts, completions, packed=None, **kwargs) -> list[float]:
    """The one that matters: the fraction of citations resolve() keeps - a [Source N] the model was shown."""
    out = []
    for c in completions:
        d = draft(c)
        if not d or not d.citations:
            out.append(0.0)
            continue
        out.append(len(resolve(d, packed).citations) / len(d.citations))
    return out

def dm(answer: str, sources: list[int], answerable: bool = True) -> str:
    return json.dumps({"answer": answer, "citations": [{"source": s, "quote": "sixty days"} for s in sources], "confidence": "high", "answerable": answerable})

ATTEMPTS = [("good", dm("Sixty days.", [1])), ("no citation", dm("Sixty days.", [])), ("invented source", dm("Sixty days.", [9]))]
for label, c in ATTEMPTS:
    f = format_reward([QUESTION], [c])[0]
    v = citation_validity_reward([QUESTION], [c], packed=PACKED)[0]
    print(f"  {f + v:.2f}  (format {f:.1f} + citations {v:.2f})  {label:16} {c[:70]}")


## Cell 5: Reward hacking - the gate item
**Read the third row of the output.** It answers nothing and scores exactly what the honest answer scores.


In [ ]:
# REWARD HACKING: the model gets full marks and you get nothing. It does not optimise your intent; it
# optimises your FUNCTION, and if the function can be satisfied without doing the work, it will be. Here is
# a completion that scores the maximum on both rewards above and answers nothing - every source it cites is
# one it was shown, so resolve() keeps them all.
GOOD = dm("Sixty days.", [1])
HACK = dm("Sixty days.", list(range(1, len(PACKED) + 1)))
EMPTY = dm("See the documents.", list(range(1, len(PACKED) + 1)))
for label, c in (("honest answer", GOOD), ("cites everything", HACK), ("says nothing, cites everything", EMPTY)):
    f = format_reward([QUESTION], [c])[0]
    v = citation_validity_reward([QUESTION], [c], packed=PACKED)[0]
    print(f"  {f + v:.2f}  {label:32}")
assert format_reward([QUESTION], [EMPTY])[0] + citation_validity_reward([QUESTION], [EMPTY], packed=PACKED)[0] == 2.0, "the hack should score full marks - that is the demonstration"
print("\n  the last row answers NOTHING and scores the same as the honest answer. A few thousand steps of this and")
print("  the model reliably says 'see the documents' with perfect citations.\n")

# The fix is not a cleverer model. It is a reward that cannot be satisfied without doing the work: the
# golden row's must_contain, through the gate's own normalise() - the same maths run_eval scores with.
MUST_CONTAIN = ["60"]      # lk-06's must_contain; on the lane the answer key IS the golden set
def grounded_answer_reward(prompts, completions, must_contain=None, **kwargs) -> list[float]:
    """Does the answer CONTAIN the fact, not just point at it? The outcome, not the shape of the outcome."""
    out = []
    for c in completions:
        d = draft(c)
        text = normalise(d.answer) if d else ""
        out.append(1.0 if d and all(normalise(m) in text for m in (must_contain or [])) else 0.0)
    return out

for label, c in (("honest answer", GOOD), ("says nothing", EMPTY)):
    total = (format_reward([QUESTION], [c])[0] + citation_validity_reward([QUESTION], [c], packed=PACKED)[0]
             + grounded_answer_reward([QUESTION], [c], must_contain=MUST_CONTAIN)[0])
    print(f"  with the grounding reward: {total:.2f}  {label}")


In [ ]:
# NOW GO LOOKING FOR YOUR OWN. Run once here so you have seen the shape of it: three candidates that score full
# marks on format and citations, and what the grounding reward does with each - because it does NOT behave
# as hoped. "It may be sixty days" contains 60 after normalise(), so the grounding reward is satisfied by an
# answer that commits to nothing: the fix has a hack of its own. And a refusal with a source scores zero on
# validity - right, until you notice the format reward still paid it for being short.
CANDIDATES = {"repeat the question back": dm("What is the notice period?", [1]),
              "hedge everything":         dm("It may be sixty days.", [1]),
              "one word":                 dm("Sixty.", [1]),
              "refuse, with a source":    dm("The provided context does not answer this question.", [1], answerable=False)}
print(f"{'candidate':28} {'format+cite':>12} {'+grounding':>11}")
for label, c in CANDIDATES.items():
    two = format_reward([QUESTION], [c])[0] + citation_validity_reward([QUESTION], [c], packed=PACKED)[0]
    three = two + grounded_answer_reward([QUESTION], [c], must_contain=MUST_CONTAIN)[0]
    print(f"  {label:26} {two:>10.2f} {three:>11.2f}")
print("\nevery fix reveals the next hack, and some fixes punish answers you would have accepted ('Sixty.').")
print("the gate for this lesson is DOCUMENT one, not ELIMINATE all - and read the top-scoring completions, not the reward curve.")


## Cell 6: Mixture of experts - 4 of 32, 19 of 20 billion


In [ ]:
# Mixture of experts, in one paragraph.
#
# A normal model runs every parameter for every token. An MoE model has many
# small "experts" and a ROUTER that picks a few of them per token. Same quality,
# far less compute per token - because most of the model sits idle each step.
#
# gpt-oss-20b, verified 2026-09-05:
EXPERTS_TOTAL = 32
EXPERTS_ACTIVE = 4
TOTAL_B = 20.0
EXPERT_B = 19.0          # the experts are 19 of the 20 billion parameters

print(f'  {EXPERTS_TOTAL} experts, {EXPERTS_ACTIVE} used per token '
      f'({EXPERTS_ACTIVE / EXPERTS_TOTAL:.0%} of them)')
print(f'  experts are {EXPERT_B}B of {TOTAL_B}B parameters '
      f'({EXPERT_B / TOTAL_B:.0%} of the model)')
print()

# That second number decides how you fine-tune it. LoRA adds a small trainable
# layer beside each matrix you target. Target the experts and you are adding a
# layer beside 95% of the model - THIRTY-TWO times over, once per expert.
def adapter_mb(target_b: float, r: int = 16) -> float:
    """Rough LoRA adapter size in MB for `target_b` billion targeted parameters.

    A rank-r LoRA adds roughly 2*r numbers per targeted dimension. The absolute
    figure is approximate; the RATIO between the two rows is the point.
    """
    return target_b * 1e9 * 2 * r * 2 / (1024 ** 2) / 4096      # bytes -> MB


print(f'  {"target":30} {"params targeted":>16} {"adapter":>10}')
for label, target in (('attention only (default)', TOTAL_B - EXPERT_B),
                      ('attention + all 32 experts', TOTAL_B)):
    print(f'  {label:30} {target:>13.1f} B {adapter_mb(target):>8.0f} MB')

print(f'  -> targeting the experts multiplies the adapter by '
      f'{adapter_mb(TOTAL_B) / adapter_mb(TOTAL_B - EXPERT_B):.0f}x')

print()
print('  Unsloth freezes the router by default, and that default is right. The')
print('  router decides WHICH expert sees a token; retraining it on a few hundred')
print('  DocuMind answers rearranges a routing scheme learned from far more data.')
print('  You wanted a citation habit, not a different model.')


## Cell 7: gpt-oss-20b, harmony, and whether it fits a free T4


In [ ]:
# Running gpt-oss-20b - and the format you cannot skip.
CODE = '''from unsloth import FastLanguageModel

# Unsloth's LINEARIZED build. No other version works for QLoRA on this model:
# gpt-oss stores its expert weights as nn.Parameter rather than nn.Linear, and
# the quantiser needs Linear layers to work with.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    max_seq_length=4096,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    # Attention only. NOT the experts - see the adapter arithmetic above.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing="unsloth",
)

# HARMONY. gpt-oss was trained on this response format and does not work
# properly without it. The tokenizer's chat template applies it for you:
messages = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": "What is the notice period?"}]
text = tokenizer.apply_chat_template(messages, tokenize=False,
                                     add_generation_prompt=True)

# If you ever call model.generate() with a raw string instead, you have skipped
# harmony, and the model will answer - just badly, in a way that looks like the
# fine-tune failed rather than like the prompt was malformed.
'''
with open('train_gpt_oss.py', 'w') as f:
    f.write(CODE)
print('train_gpt_oss.py written')
print()

# Does it fit the free tier this course has used since 10.5?
T4_USABLE = 14.5        # a 16 GB T4, minus CUDA and the driver
for label, need in (('gpt-oss-20b QLoRA (Unsloth)', 14.0),
                    ('gpt-oss-20b QLoRA (newer Unsloth)', 12.8),
                    ('gpt-oss-20b, any other method', 65.0)):
    room = T4_USABLE - need
    verdict = ('DOES NOT FIT' if room < 0 else
               'fits, with nothing to spare' if room < 1 else
               'fits, barely' if room < 3 else 'fits')
    print(f'  {label:34} {need:>5.1f} GB   {verdict}')

print()
print('  This is the first lesson where the free T4 is genuinely marginal. 14 GB')
print('  against 14.5 GB usable leaves no room for a longer sequence or a bigger')
print('  batch, so budget an L4 - the same card lesson 11.4 rents for serving.')
print('  Note the last row: without Unsloth this model needs 65 GB. That is the')
print('  difference between a lesson and a research grant.')


## Cell 8: The GGUF stop-token bug, and the kit's fix


In [ ]:
sys.path.insert(0, f"{KIT}/deploy/services/slm")
from make_modelfile import build

# THE BUG THAT WILL COST YOU AN AFTERNOON. You fine-tune, export a GGUF, load it in Ollama, and the model
# never stops talking - or answers in the wrong voice. The GGUF carried the WEIGHTS but not the CHAT
# TEMPLATE and STOP TOKENS; the weights learned harmony (gpt-oss) or Gemma's turns, and the runtime is
# feeding them something else. The fix on the lane is not a Modelfile typed from memory: it is
# services/slm/make_modelfile.py, which renders the template FROM THE TOKENIZER you trained with and takes
# the stop tokens from its special tokens. Offline, build() shows what it writes for a harmony template.
print(build("<|start|>user<|message|>{{ .Prompt }}<|end|>\n<|start|>assistant<|message|>", ["<|end|>", "<|return|>"], "documind-slm.gguf"))
print("How to tell the failures apart, quickly:")
print("  never stops          -> stop tokens missing")
print("  wrong voice/format   -> chat template wrong or absent")
print("  right voice, wrong facts -> actually your fine-tune. Now you can debug it.")
print("\n10.5 generates the real one: python services/slm/make_modelfile.py --model documind-slm-lora --gguf documind-slm.gguf")


## Cell 9: Did any of this help? The instruments the lane has


In [ ]:
# DID ANY OF THIS HELP? Not "is it good" - "is it BETTER than gemini-3.6-flash, on our questions, for our
# money". On the lane that question has two instruments and no stand-ins:
#   a tuned Gemini  -> make candidate + make eval-live API=<candidate> (the gate) + make judge API_B=<candidate> (pairwise, 10.4)
#   the SLM         -> services/slm/compare_backends.py through the gateway (11.3/11.4): the same golden rows, retrieval ONCE
#                      per question through the one retrieve(), every backend answering from the SAME context; groundedness,
#                      citation precision, p95 and rupees per thousand - and the SLM's rupees are a RATE, not a price.
# Its maths is proven offline, on a fixture that can go red:
r = subprocess.run([sys.executable, f"{KIT}/deploy/services/slm/compare_backends.py", "--selftest"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "compare_backends --selftest failed: the summary maths is wrong"
print("live, once 11.4 has deployed the SLM behind the gateway:")
print("  LITELLM_URL=... python services/slm/compare_backends.py --rows 20 > compare.csv && python services/slm/compare_backends.py --summary compare.csv")
print("read the groundedness and the refusal rows first, and never quote the SLM's Rs/1k without the volume it holds at.")


## Cell 10: Look how far


In [ ]:
# Module 10, end to end - on the lane.
STAGES = [("10.1", "a dataset from the corpus, the golden set excluded; a tuned Gemini behind GENERATOR_MODEL; the gate twice"),
          ("10.2", "the API's cache manager finally called: the pack, the version, cached_tokens in the row"),
          ("10.3", "the dataset build as the batch job; router.py finally on the request path, priced per tier"),
          ("10.4", "the second judge on the lane's own answers; two judges compared; trajectories from real tool_calls"),
          ("10.5", "the same file, a model you keep; DLP live; a GGUF under the name 11.4 builds from"),
          ("10.6", "rewards written on the contract, resolve() as the citation judge, the hack, the honest comparison")]
for lid, what in STAGES:
    print(f"  {lid}  {what}")
print("\n  the through-line: the dataset is a document, the model is a setting, the gate is the judge - and every")
print("  step was a cost or a quality decision with a number from the lane attached. None was 'use a bigger model'.")
print("\n  Module 11 runs it: Ollama on Cloud Run with an L4, behind the gateway, priced against the model you started with.")


## Done — Module 10 is complete
You chose between distillation and GRPO on one question, masked a prompt so the model learns answers rather than questions, wrote rewards on the lane's contract with `resolve()` as the citation judge, watched a worthless completion earn full marks and a hedge beat the fix, computed why LoRA on MoE experts inflates an adapter, found that gpt-oss-20b needs 14 GB against 65 without Unsloth, learned which of three Ollama symptoms is your fine-tune, and named the two instruments that decide whether any of it helped.

**Next:** Module **11** runs it. Lesson **11.4** deploys the GGUF as Ollama on Cloud Run with an L4 GPU behind the gateway from 11.3, and `compare_backends.py` prints the comparison; **11.5** asks whether Cloud Run or GKE Autopilot should host it.
